# 02 - Silver Transformações

Objetivo: ler a camada Bronze local e gerar uma versão Silver com padronização básica, tratamento de tipos, remoção de duplicidades e organização dos dados.


In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd


In [ ]:
# Localiza automaticamente o arquivo Bronze mais recente
bronze_base = Path("../data/bronze/uf_alfabetizacao")

arquivos = sorted(bronze_base.glob("execution_date=*/uf_alfabetizacao.parquet"))

if not arquivos:
    raise FileNotFoundError("Nenhum arquivo Bronze encontrado. Execute primeiro o notebook 01.")

bronze_file = arquivos[-1]

print("Arquivo Bronze usado:", bronze_file)

df = pd.read_parquet(bronze_file)
df.head()


In [ ]:
# Visão geral dos dados
print("Linhas:", len(df))
print("Colunas:", len(df.columns))

df.info()


In [ ]:
# Padronização básica de nomes e tipos
df_silver = df.copy()

# Padroniza textos
colunas_texto = ["sigla_uf", "sigla_uf_nome", "serie", "rede"]

for coluna in colunas_texto:
    if coluna in df_silver.columns:
        df_silver[coluna] = df_silver[coluna].astype("string").str.strip()

# Tipos numéricos
colunas_numericas = [
    "taxa_alfabetizacao",
    "media_portugues",
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
]

for coluna in colunas_numericas:
    if coluna in df_silver.columns:
        df_silver[coluna] = pd.to_numeric(df_silver[coluna], errors="coerce")

if "ano" in df_silver.columns:
    df_silver["ano"] = pd.to_numeric(df_silver["ano"], errors="coerce").astype("Int64")

# Remove duplicidades completas
df_silver = df_silver.drop_duplicates()

df_silver.head()


In [ ]:
# Regras simples de consistência
print("Nulos por coluna:")
display(df_silver.isna().sum())

print("Duplicados:", df_silver.duplicated().sum())

print("UFs:", sorted(df_silver["sigla_uf"].dropna().unique()))


In [ ]:
execution_date = datetime.now().strftime("%Y-%m-%d")

silver_path = Path(f"../data/silver/uf_alfabetizacao/execution_date={execution_date}")
silver_path.mkdir(parents=True, exist_ok=True)

silver_file = silver_path / "uf_alfabetizacao_silver.parquet"

df_silver.to_parquet(silver_file, index=False)

print(f"Arquivo Silver salvo em: {silver_file}")
